# Math Keyword Extraction - Interactive Testing

This notebook demonstrates how to use the custom NER model to extract mathematical keywords from user queries and retrieve relevant content.

## 1. Setup and Import

In [4]:
# Import the custom extractor
from math_keyword_extractor import MathKeywordExtractor, SQLRetriever
import pandas as pd
import json

## 2. Initialize the System

In [5]:
# Initialize the keyword extractor
extractor = MathKeywordExtractor(
    keywords_file="probability_keywords.csv",
    model_path="math_ner_model"
)

# Load or train the model
if not extractor.load_model():
    print("Training new NER model...")
    training_data = extractor.generate_training_data(num_examples=500)
    extractor.train_ner_model(training_data, n_iter=10)
    print("Model training completed!")
else:
    print("Loaded existing model successfully!")

# Initialize the retriever
retriever = SQLRetriever(db_path="knowledge_base.db")

print(f"\nLoaded {len(extractor.keywords)} mathematical keywords")
print("System ready for testing!")

Loaded 467 keywords from probability_keywords.csv
Loaded model from math_ner_model
Loaded existing model successfully!

Loaded 467 mathematical keywords
System ready for testing!


## 3. Test Individual Queries

In [6]:
def test_query(query_text, show_details=True):
    """
    Test a single query and display results
    """
    print(f"\n🔍 Query: '{query_text}'")
    print("="*60)
    
    # Extract keywords using both NER and fuzzy matching
    keywords = extractor.extract_keywords(query_text, top_k=5)
    
    if keywords:
        print(f"✅ Extracted {len(keywords)} keywords:")
        for i, (keyword, confidence) in enumerate(keywords, 1):
            print(f"   {i}. '{keyword}' (confidence: {confidence:.3f})")
        
        # Get just the keyword strings for retrieval
        keyword_strings = [kw[0] for kw in keywords]
        
        # Retrieve relevant chunks
        chunks = retriever.retrieve_chunks(keyword_strings, top_k=3)
        
        if chunks:
            print(f"\n📚 Retrieved {len(chunks)} relevant chunks:")
            for i, chunk in enumerate(chunks, 1):
                print(f"\n   📖 Chunk {i}:")
                print(f"      Topic: {chunk['topic']}")
                print(f"      Content: {chunk['content'][:150]}...")
                print(f"      Matched keyword: '{chunk['matched_keyword']}'")
                if show_details:
                    metadata = json.loads(chunk['metadata']) if chunk['metadata'] else {}
                    print(f"      Metadata: {metadata}")
        else:
            print("\n❌ No relevant chunks found in database")
    else:
        print("❌ No keywords extracted from query")
    
    return keywords, chunks if keywords else []

In [7]:
# Test some example queries
test_queries = [
    "I want to learn about conditional probability",
    "Can you explain Bayes theorem and how it works?",
    "What's the difference between variance and standard deviation?",
    "Help me understand normal distribution",
    "I'm studying hypothesis testing and p-values"
]

for query in test_queries:
    test_query(query)


🔍 Query: 'I want to learn about conditional probability'
✅ Extracted 5 keywords:
   1. 'probability' (confidence: 0.800)
   2. 'conditional probability' (confidence: 0.790)
   3. 'conditional pmf' (confidence: 0.693)
   4. 'marginal probability' (confidence: 0.692)
   5. 'joint probability' (confidence: 0.679)

📚 Retrieved 3 relevant chunks:

   📖 Chunk 1:
      Topic: conditional probability
      Content: Conditional probability is the probability of an event given that another event has occurred......
      Matched keyword: 'probability'


JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

## 4. Interactive Testing

Use this cell to test your own queries:

In [8]:
# Test your own query here!
your_query = "I need help with central limit theorem and sampling distributions"

keywords, chunks = test_query(your_query, show_details=True)


🔍 Query: 'I need help with central limit theorem and sampling distributions'
✅ Extracted 5 keywords:
   1. 'central limit theorem' (confidence: 0.860)
   2. 'sampling distribution' (confidence: 0.800)
   3. 't distribution' (confidence: 0.743)
   4. 'f distribution' (confidence: 0.743)
   5. 'sampling distributions' (confidence: 0.700)

❌ No relevant chunks found in database


## 5. Compare NER vs Fuzzy Matching

In [9]:
def compare_extraction_methods(query_text):
    """
    Compare NER-only vs Fuzzy-only vs Combined extraction
    """
    print(f"\n🔬 Comparing extraction methods for: '{query_text}'")
    print("="*70)
    
    # NER only
    ner_keywords = extractor.extract_keywords(query_text, use_ner=True, use_fuzzy=False, top_k=5)
    print(f"\n🤖 NER Only ({len(ner_keywords)} keywords):")
    for keyword, conf in ner_keywords:
        print(f"   • '{keyword}' ({conf:.3f})")
    
    # Fuzzy only
    fuzzy_keywords = extractor.extract_keywords(query_text, use_ner=False, use_fuzzy=True, top_k=5)
    print(f"\n🔍 Fuzzy Only ({len(fuzzy_keywords)} keywords):")
    for keyword, conf in fuzzy_keywords:
        print(f"   • '{keyword}' ({conf:.3f})")
    
    # Combined
    combined_keywords = extractor.extract_keywords(query_text, use_ner=True, use_fuzzy=True, top_k=5)
    print(f"\n🔥 Combined ({len(combined_keywords)} keywords):")
    for keyword, conf in combined_keywords:
        print(f"   • '{keyword}' ({conf:.3f})")
    
    return ner_keywords, fuzzy_keywords, combined_keywords

In [19]:
# Test comparison with different query types
test_queries = [
    "I want to learn about conditional probability",  # Exact match
    "Can you explain conditionl probabilty?",         # Typo
    "What is the Bayesian approach?",                 # Partial match
    "Help with hypothesis testing",                   # Short query
]

for query in test_queries:
    compare_extraction_methods(query)


🔬 Comparing extraction methods for: 'I want to learn about conditional probability'

🤖 NER Only (1 keywords):
   • 'conditional probability' (0.700)

🔍 Fuzzy Only (5 keywords):
   • 'probability' (0.800)
   • 'conditional probability' (0.800)
   • 'conditional pmf' (0.693)
   • 'marginal probability' (0.692)
   • 'joint probability' (0.679)

🔥 Combined (5 keywords):
   • 'probability' (0.800)
   • 'conditional probability' (0.790)
   • 'conditional pmf' (0.693)
   • 'marginal probability' (0.692)
   • 'joint probability' (0.679)

🔬 Comparing extraction methods for: 'Can you explain conditionl probabilty?'

🤖 NER Only (1 keywords):
   • 'conditionl probabilty' (0.700)

🔍 Fuzzy Only (5 keywords):
   • 'conditional probability' (0.747)
   • 'probability' (0.727)
   • 'joint probability' (0.650)
   • 'transition probability' (0.648)
   • 'conditional pmf' (0.640)

🔥 Combined (5 keywords):
   • 'conditional probability' (0.747)
   • 'probability' (0.727)
   • 'conditionl probabilty' (0.700

## 6. Test with Typos and Variations

In [20]:
# Test robustness with typos and variations
typo_queries = [
    "What is conditionl probabilty?",           # Typos in "conditional probability"
    "Explain baye's theorm",                    # Typos in "bayes theorem"
    "I need help with normall distribution",   # Typo in "normal distribution"
    "What's the diferrence between variance and stdev?",  # Multiple typos
    "Can you teach me about expected val?",     # Abbreviated form
    "I'm studying monte carlo simulations",     # Different spacing
]

print("🛠️ Testing robustness with typos and variations:")
print("="*60)

for query in typo_queries:
    keywords = extractor.extract_keywords(query, top_k=3)
    print(f"\nQuery: '{query}'")
    if keywords:
        print(f"✅ Extracted: {[(kw, f'{conf:.2f}') for kw, conf in keywords]}")
    else:
        print("❌ No keywords extracted")

🛠️ Testing robustness with typos and variations:

Query: 'What is conditionl probabilty?'
✅ Extracted: [('conditional probability', '0.75'), ('probability', '0.73'), ('conditionl probabilty', '0.70')]

Query: 'Explain baye's theorm'
✅ Extracted: [('bayes theorem', '0.70')]

Query: 'I need help with normall distribution'
✅ Extracted: [('t distribution', '0.77'), ('f distribution', '0.77'), ('normal distribution', '0.76')]

Query: 'What's the diferrence between variance and stdev?'
✅ Extracted: [('variance', '0.72'), ('covariance', '0.64'), ('stdev', '0.60')]

Query: 'Can you teach me about expected val?'
✅ Extracted: [('expected value', '0.71'), ('expected val', '0.70')]

Query: 'I'm studying monte carlo simulations'
✅ Extracted: [('monte carlo simulations', '0.80'), ('monte carlo simulation', '0.73')]


## 7. Keyword Coverage Analysis

In [21]:
# Analyze what keywords are available
print(f"📊 Available Keywords Analysis")
print("="*40)
print(f"Total keywords: {len(extractor.keywords)}")

# Show some sample keywords
print("\n📝 Sample keywords (first 20):")
for i, keyword in enumerate(extractor.keywords[:20], 1):
    print(f"   {i:2d}. {keyword}")

# Keywords by category (simple analysis)
probability_keywords = [kw for kw in extractor.keywords if 'probability' in kw.lower()]
distribution_keywords = [kw for kw in extractor.keywords if 'distribution' in kw.lower()]
test_keywords = [kw for kw in extractor.keywords if 'test' in kw.lower()]

print(f"\n📈 Keyword categories:")
print(f"   • Probability-related: {len(probability_keywords)}")
print(f"   • Distribution-related: {len(distribution_keywords)}")
print(f"   • Test-related: {len(test_keywords)}")

print(f"\n🎯 Example probability keywords:")
for kw in probability_keywords[:5]:
    print(f"   • {kw}")

📊 Available Keywords Analysis
Total keywords: 467

📝 Sample keywords (first 20):
    1. probability
    2. random variable
    3. sample space
    4. event space
    5. conditional probability
    6. joint probability
    7. marginal probability
    8. probability distribution
    9. probability mass function
   10. probability density function
   11. cumulative distribution function
   12. normal distribution
   13. Gaussian distribution
   14. uniform distribution
   15. Bernoulli distribution
   16. binomial distribution
   17. Poisson distribution
   18. exponential distribution
   19. geometric distribution
   20. hypergeometric distribution

📈 Keyword categories:
   • Probability-related: 60
   • Distribution-related: 45
   • Test-related: 13

🎯 Example probability keywords:
   • probability
   • conditional probability
   • joint probability
   • marginal probability
   • probability distribution


## 8. Batch Testing

In [30]:
def batch_test(queries_list):
    """
    Test multiple queries and summarize results
    """
    results = []
    
    for query in queries_list:
        keywords = extractor.extract_keywords(query, top_k=3)
        keyword_strings = [kw[0] for kw in keywords] if keywords else []
        chunks = retriever.retrieve_chunks(keyword_strings, top_k=2) if keywords else []
        
        results.append({
            'query': query,
            'keywords_found': len(keywords),
            'chunks_retrieved': len(chunks),
            'top_keyword': keywords[0][0] if keywords else 'None',
            'top_confidence': keywords[0][1] if keywords else 0.0
        })
    
    # Convert to DataFrame for easy viewing
    df = pd.DataFrame(results)
    return df

# Test with various query types
test_queries = [
    "What is conditional probability?",
    "Explain Bayes theorem",
    "Normal distribution properties",
    "Expected value calculation",
    "Hypothesis testing steps",
    "Central limit theorem",
    "What is variance?",
    "Joint probability distribution",
    "Random variable definition",
    "Monte Carlo simulation",
    "explain event probability like im 5",
    "explain bayes theorem in simple terms"
]

results_df = batch_test(test_queries)
print("📊 Batch Test Results:")
print("="*50)
print(results_df.to_string(index=False))

print(f"\n📈 Summary Statistics:")
print(f"   • Average keywords per query: {results_df['keywords_found'].mean():.1f}")
print(f"   • Average chunks retrieved: {results_df['chunks_retrieved'].mean():.1f}")
print(f"   • Average confidence: {results_df['top_confidence'].mean():.3f}")
print(f"   • Queries with results: {len(results_df[results_df['keywords_found'] > 0])}/{len(results_df)}")

📊 Batch Test Results:
                                query  keywords_found  chunks_retrieved                    top_keyword  top_confidence
     What is conditional probability?               3                 2                    probability        0.800000
                Explain Bayes theorem               1                 1                  bayes theorem        0.768000
       Normal distribution properties               3                 1            normal distribution        0.800000
           Expected value calculation               3                 1     expected value calculation        0.800000
             Hypothesis testing steps               3                 0             hypothesis testing        0.800000
                Central limit theorem               1                 0          central limit theorem        0.800000
                    What is variance?               2                 0                       variance        0.720000
       Joint probability d

## 9. Your Custom Test Area

Use the cells below to test your own specific queries:

In [28]:
# Enter your custom query here
my_query = "show me an example of mean of 10 numbers"

if my_query:
    test_query(my_query, show_details=True)
else:
    print("Please enter a query in the my_query variable above")


🔍 Query: 'show me an example of mean of 10 numbers'
✅ Extracted 2 keywords:
   1. 'mean of' (confidence: 0.700)
   2. '10 numbers' (confidence: 0.700)

❌ No relevant chunks found in database


In [ ]:
# Test multiple custom queries at once
my_queries = [
    # Add your queries here, for example:
    # "How do I calculate confidence intervals?",
    # "What's the difference between Type I and Type II errors?",
    # "Explain the law of large numbers",
]

if my_queries:
    for query in my_queries:
        test_query(query)
else:
    print("Add your queries to the my_queries list above")

## 10. Model Performance Tips

### To improve the model:
1. **Add more training data**: Increase `num_examples` in training
2. **Train longer**: Increase `n_iter` for more training iterations
3. **Adjust fuzzy threshold**: Modify `extractor.fuzzy_threshold` (default: 80)
4. **Add more keywords**: Expand the `probability_keywords.csv` file

### The system works best when:
- Queries contain mathematical terminology from the keyword list
- Keywords are spelled reasonably correctly (fuzzy matching helps with minor typos)
- Queries are focused on specific mathematical concepts rather than very general questions